### **Importing the dependecies**

In [122]:
import re
import pandas
import numpy

### **Simple Tokenization**

In [75]:
with open(r"C:\Users\Subham Pathak\Desktop\AI\LLMs\dataset\raw_text.txt", 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [76]:
print(f"The length of the text is : {len(raw_text)}")

The length of the text is : 581887


In [77]:
raw_text = raw_text.lower()

In [78]:
test1 = re.split(r'([.,:;?_!"()\'] |--|\s)', raw_text)
print(test1[:150])

['\ufeffproject', ' ', "gutenberg's", ' ', 'the', ' ', 'adventures', ' ', 'of', ' ', 'sherlock', ' ', 'holmes', ', ', 'by', ' ', 'arthur', ' ', 'conan', ' ', 'doyle', '\n', '', '\n', 'this', ' ', 'ebook', ' ', 'is', ' ', 'for', ' ', 'the', ' ', 'use', ' ', 'of', ' ', 'anyone', ' ', 'anywhere', ' ', 'at', ' ', 'no', ' ', 'cost', ' ', 'and', ' ', 'with', '\n', 'almost', ' ', 'no', ' ', 'restrictions', ' ', 'whatsoever', '. ', '', ' ', 'you', ' ', 'may', ' ', 'copy', ' ', 'it', ', ', 'give', ' ', 'it', ' ', 'away', ' ', 'or', '\n', 're-use', ' ', 'it', ' ', 'under', ' ', 'the', ' ', 'terms', ' ', 'of', ' ', 'the', ' ', 'project', ' ', 'gutenberg', ' ', 'license', ' ', 'included', '\n', 'with', ' ', 'this', ' ', 'ebook', ' ', 'or', ' ', 'online', ' ', 'at', ' ', 'www.gutenberg.net', '\n', '', '\n', '', '\n', 'title', ': ', 'the', ' ', 'adventures', ' ', 'of', ' ', 'sherlock', ' ', 'holmes', '\n', '', '\n', 'author', ': ', 'arthur', ' ', 'conan', ' ', 'doyle', '\n', '', '\n', 'release', ' '

In [79]:
preprocessed = [item.strip() for item in test1 if item.strip()]

In [80]:
print(preprocessed[:150])

['\ufeffproject', "gutenberg's", 'the', 'adventures', 'of', 'sherlock', 'holmes', ',', 'by', 'arthur', 'conan', 'doyle', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '.', 'you', 'may', 'copy', 'it', ',', 'give', 'it', 'away', 'or', 're-use', 'it', 'under', 'the', 'terms', 'of', 'the', 'project', 'gutenberg', 'license', 'included', 'with', 'this', 'ebook', 'or', 'online', 'at', 'www.gutenberg.net', 'title', ':', 'the', 'adventures', 'of', 'sherlock', 'holmes', 'author', ':', 'arthur', 'conan', 'doyle', 'release', 'date', ':', 'november', '29', ',', '2002', '[ebook', '#1661]', 'last', 'updated', ':', 'may', '20', ',', '2019', 'language', ':', 'english', 'character', 'set', 'encoding', ':', 'utf-8', '***', 'start', 'of', 'this', 'project', 'gutenberg', 'ebook', 'the', 'adventures', 'of', 'sherlock', 'holmes', '***', 'produced', 'by', 'an', 'anonymous', 'project', 'gutenberg', 'volun

In [81]:
all_words = (sorted(set(preprocessed)))

In [82]:
vocab = {token: integer for integer, token in enumerate(all_words)}

In [105]:
len(vocab)

11699

##### **Creating Class for encoding and decoding part of tokenization**

In [84]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {s:i for i,s in vocab.items()}
        
    def encode(self, text):
        text = text.lower()
        preprocessed = re.split(r'([,.:;/?"\!-_*^%$#@~`]|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[id] for id in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\;'':])', r'\1', text)
        return text
        

In [85]:
token = SimpleTokenizer(vocab)

text = '''
“My dear Holmes said I “this is too much. You would certainly have
    '''  

In [86]:
ids = token.encode(text)
print(ids)

[11433, 2568, 4831, 8340, 5001, 11529, 5375, 9988, 6461, 38, 11107, 11018, 1660, 4645]


In [87]:
token.decode(ids)

'“my dear holmes said i “this is too much. you would certainly have'

In [112]:
# Modifying the tokenizer

len(preprocessed)

118945

In [99]:
all_tokens = sorted(preprocessed)
all_tokens.extend(['<|unk|>', '<|endoftext|>'])  
  

In [111]:
type(all_tokens)
len(all_tokens)

118947

In [101]:
vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [104]:
len(vocab)

11699

In [ ]:
print(vocab.items())

In [121]:
#Creating the version 2 (modified version: handling error if unknown token is recognized)

class SimpleTokenizerV2:
    
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {integer:token for integer, token in enumerate(vocab.items())}
        
    def encode(self, text):
        text = text.lower()
        
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [items.strip() for items in preprocessed if items.strip()]
        
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>" for item in preprocessed
        ]
        
        ids = [self.str_to_int[id] for id in preprocessed]
        return ids
    
    def decode(self, ids):
        sentence = " ".join([self.int_to_str[id] for id in ids])
        sentence = re.sub(r'\s+([,.;:!_()\])', r'\1', sentence)
        return sentence

### **Byte Pair Encoding**